# 4 - Indici di Posizione con R

Questo notebook copre gli **indici di posizione** (o di tendenza centrale):
1. **Moda** — valore più frequente
2. **Minimo e Massimo** — valori estremi
3. **Mediana** — valore centrale
4. **Quartili e Percentili** — suddivisione in parti uguali
5. **Media Aritmetica** — somma diviso il numero di osservazioni
6. **Valori Anomali** — effetto di outlier su media e mediana
7. **Indici in Classi** — distribuzione di frequenza per classi
8. **Media Ponderata** — media con pesi
9. **Media Geometrica** — per tassi di crescita
10. **Media Armonica** — per velocità e rapporti

---
## Carichiamo il dataset Iris

Usiamo il dataset `iris` (già disponibile in R) che contiene misure di sepali e petali di tre specie di fiori.

In [ ]:
# Carichiamo il dataset iris
data("iris")

# Mostriamo le prime righe
head(iris)

# Attach rende le variabili del dataframe accessibili direttamente
attach(iris)

---
## 1. Moda

La **moda** è il valore che compare con maggiore frequenza. Per variabili qualitative (come `Species`) si usa la tabella delle frequenze.

In [ ]:
# Moda: tabella delle frequenze per Species
table(Species)

---
## 2. Minimo e Massimo

Il **minimo** e il **massimo** sono i valori estremi della distribuzione.

In [ ]:
# Minimo e massimo di Petal.Length
min(Petal.Length)
max(Petal.Length)

# Numero di osservazioni
n <- length(Petal.Length)
n

---
## 3. Mediana

La **mediana** è il valore che divide la distribuzione in due parti uguali (50° percentile).
Se n è pari, la mediana è la media dei due valori centrali.

In [ ]:
# n/2 = 75, quindi la mediana è tra il 75° e 76° valore ordinato
n/2

# Valori centrali ordinati
sort(Petal.Length)[c(75, 76)]

# Mediana con funzione built-in
median(Petal.Length)

---
## 4. Quartili e Percentili

I **quartili** dividono la distribuzione in 4 parti uguali:
- **Q1** (primo quartile) = 25° percentile
- **Q2** (secondo quartile) = 50° percentile = mediana
- **Q3** (terzo quartile) = 75° percentile

I **percentili** dividono in 100 parti. I **decili** in 10 parti.

In [ ]:
# Primo quartile: n/4 = 37.5, prendiamo il 38° valore
n/4
sort(Petal.Length)[c(38)]

# Terzo quartile: 3n/4 = 112.5, prendiamo il 113° valore
n/4 * 3
sort(Petal.Length)[c(113)]

In [ ]:
# Quantili in generale (0%, 25%, 50%, 75%, 100%)
quantile(Petal.Length)

# Decili (da 0% a 100% con step del 10%)
quantile(Petal.Length, seq(0, 1, 0.1))

# Percentili (da 0% a 100% con step dell'1%)
# View(quantile(Petal.Length, seq(0, 1, 0.01)))

---
## 5. Media Aritmetica

La **media aritmetica** è la somma di tutti i valori divisa per il numero di osservazioni.

In [ ]:
# Media calcolata manualmente
sum(Petal.Length) / n

# Media con funzione built-in
mean(Petal.Length)

---
## 6. Effetto dei Valori Anomali (Outlier)

La **media** è sensibile ai valori anomali, mentre la **mediana** è robusta (resistente).
Vediamo l'effetto aggiungendo valori estremi.

In [ ]:
# Aggiungiamo un valore anomalo (31)
c(Petal.Length, 31)

# Media con un outlier estremo (541)
mean(c(Petal.Length, 541))
median(c(Petal.Length, 541))

# Media con due outlier estremi
mean(c(Petal.Length, 541, 378))
median(c(Petal.Length, 541, 378))

---
## 7. Indici di Posizione in Classi

Quando i dati sono raggruppati in **classi di intervallo**, costruiamo una distribuzione di frequenze.
Usiamo `cut()` per creare le classi.

In [ ]:
# Creiamo classi di ampiezza 1 per Petal.Length (da 0 a 7)
Petal.Length_cl <- cut(Petal.Length, seq(0, 7, 1))

# Distribuzione di frequenze completa
distr_freq <- as.data.frame(
  cbind(
    ni = table(Petal.Length_cl),           # frequenze assolute
    fi = table(Petal.Length_cl) / n,       # frequenze relative
    Ni = cumsum(table(Petal.Length_cl)),   # frequenze cumulate assolute
    Fi = cumsum(table(Petal.Length_cl)) / n  # frequenze cumulate relative
  )
)

distr_freq

In [ ]:
# Quantili
quantile(Petal.Length)

# Troviamo la classe che contiene il primo quartile (Fi > 0.25)
which(distr_freq$Fi > 0.25)[1]

In [ ]:
# Summary: un riepilogo completo (min, Q1, mediana, media, Q3, max)
summary(Petal.Length)

# Summary dell'intero dataset
summary(iris)

---
## 8. Media Ponderata

La **media ponderata** assegna un peso diverso a ciascun valore.
Nel caso di dati in classi, usiamo il **centro di classe** (`xCi`) come valore rappresentativo.

In [ ]:
# Aggiungiamo il centro di classe (xCi)
distr_freq$xCi <- seq(0.5, 6.5, 1)

# Prodotto centro di classe * frequenza assoluta
distr_freq$xCi * distr_freq$ni

# Media ponderata manuale: somma(xCi * ni) / somma(ni)
sum(distr_freq$xCi * distr_freq$ni) / sum(distr_freq$ni)

# Media ponderata con funzione built-in
weighted.mean(distr_freq$xCi, distr_freq$ni)

---
## 9. Media Geometrica

La **media geometrica** è utile per tassi di crescita, rendimenti, percentuali.
Si calcola come: $\sqrt[n]{x_1 \cdot x_2 \cdot ... \cdot x_n}$

Esempio: numero di cellule in 5 giorni consecutivi.

In [ ]:
# Dati: numero di cellule in 5 giorni
cellule <- c(1000, 1800, 2100, 3000, 5000)

# Calcoliamo gli incrementi percentuali giornalieri
# install.packages("quantmod")  # se necessario
library(quantmod)
incrementi <- quantmod::Delt(cellule) * 100
incrementi <- incrementi[-1, 1]  # rimuoviamo il primo NA
incrementi

In [ ]:
# Funzione per la media geometrica
geometric.mean <- function(x) {
  return(prod(x)^(1 / length(x)))
}

# Media geometrica degli incrementi
geometric.mean(incrementi)

# Confronto con la media aritmetica
mean(incrementi)

---
## 10. Media Armonica

La **media armonica** è usata per grandezze come velocità medie.
Si calcola come: $\frac{n}{\sum_{i=1}^{n} \frac{1}{x_i}}$

Esempio: velocità in 4 tratti di un percorso.

In [ ]:
# Velocità in 4 tratti (km/h)
speed <- c(100, 80, 40, 90)

# Reciproci delle velocità
1 / speed

# Media armonica: 1 / media(1/speed)
1 / mean(1 / speed)

In [ ]:
# Funzione per la media armonica
armonic.mean <- function(x) {
  1 / (sum(1 / x) / length(x))
}

armonic.mean(speed)

# Confronto con la media aritmetica
mean(speed)